# Core workflow / 核心工作流

This notebook parses a small CDXML graph, navigates typed wrappers, edits it, and verifies both string and file round trips.

本示例解析一个小型 CDXML 图，演示类型化导航与修改，并验证字符串和文件往返。

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

from cdxml_om import BondOrder, CDXMLDocument, Node, Point2D

source = """<CDXML><page id="1"><fragment id="2">
  <n id="3" Element="6" p="1.25 2.5"/>
  <n id="4" Element="8" p="3.5 4.75"/>
  <b id="5" B="3" E="4" Order="1"/>
</fragment></page></CDXML>"""
document = CDXMLDocument.from_string(source)
page = document.pages[0]
fragment = page.fragments[0]
carbon = document.get(Node, 3)
oxygen = document.get(Node, 4)
assert carbon is not None and oxygen is not None
assert carbon is fragment.nodes[0]
assert carbon is document.find(Node)[0]
assert carbon.position == Point2D(1.25, 2.5)
assert fragment.bonds[0].order == BondOrder.SINGLE
print(f"Parsed {len(document.pages)} page, {len(fragment.nodes)} nodes, {len(fragment.bonds)} bond")

Parsed 1 page, 2 nodes, 1 bond


## Edit and validate / 修改与验证

Wrappers for the same XML element are identity-stable. New document objects receive IDs, while removal affects only the selected collection item.

同一个 XML 元素对应的 wrapper 身份稳定。新建文档对象会分配 ID；移除只影响所选集合成员。

In [2]:
bond = fragment.bonds[0]
bond.order = BondOrder.DOUBLE
oxygen.charge = -1
oxygen.position = Point2D(4.0, 5.0)
created_node = fragment.nodes.create(element=7, position=(6.0, 8.0))
created_bond = fragment.bonds.create(begin=oxygen, end=created_node, order=BondOrder.SINGLE)
created_node_id = created_node.id
created_bond_id = created_bond.id
assert created_node_id is not None and created_bond_id is not None
temporary_node = fragment.nodes.create(element=7, position=(6.0, 8.0))
temporary_id = temporary_node.id
assert temporary_id is not None
assert document.get(Node, temporary_id) is temporary_node
fragment.nodes.remove(temporary_node)
assert document.get(Node, temporary_id) is None
assert bond.begin is carbon and bond.end is oxygen
assert created_bond.begin is oxygen and created_bond.end is created_node
assert document.validate().is_valid
print(f"Changed bond order to {bond.order.name}; oxygen position is {oxygen.position}")

Changed bond order to DOUBLE; oxygen position is Point2D(x=4.0, y=5.0)


## Serialize and reload / 序列化并重新加载

The object wrappers belong to one document instance; after reloading, references resolve to wrappers owned by the new document.

对象 wrapper 属于各自的文档实例；重新加载后，引用会解析为新文档所拥有的 wrapper。

In [3]:
serialized = document.to_string()
reloaded = CDXMLDocument.from_string(serialized)
reloaded_fragment = reloaded.pages[0].fragments[0]
reloaded_bond = reloaded_fragment.bonds[0]
assert reloaded_bond.begin is reloaded_fragment.nodes[0]
assert reloaded_bond.end is reloaded_fragment.nodes[1]
assert reloaded_bond.order == BondOrder.DOUBLE
assert reloaded_fragment.nodes[1].charge == -1
reloaded_created_node = reloaded.get(Node, created_node_id)
assert reloaded_created_node is reloaded_fragment.nodes[2]
reloaded_created_bond = reloaded.get(type(created_bond), created_bond_id)
assert reloaded_created_bond is reloaded_fragment.bonds[1]
assert reloaded_created_bond.begin is reloaded_fragment.nodes[1]
assert reloaded_created_bond.end is reloaded_created_node
assert reloaded_fragment.nodes[1].position == Point2D(4.0, 5.0)
assert reloaded.validate().is_valid
with TemporaryDirectory(prefix="cdxml-om-example-") as temporary_directory:
    path = Path(temporary_directory) / "edited.cdxml"
    document.to_file(path)
    from_file = CDXMLDocument.from_file(path)
    assert from_file.validate().is_valid
    assert from_file.pages[0].fragments[0].bonds[1].end.id == created_node_id
print(f"Reloaded {len(reloaded_fragment.nodes)} nodes and {len(reloaded_fragment.bonds)} bonds")
print("Created node/bond identities, typed edits, and references survive both round trips.")
print("CDXML-OM notebook: 01_core_workflow OK")

Reloaded 3 nodes and 2 bonds
Created node/bond identities, typed edits, and references survive both round trips.
CDXML-OM notebook: 01_core_workflow OK
